In [ ]:
"""
MADALINE Heart Disease Classification and Adaptive Decision Analysis System
---------------------------------------------------------------------------
Dataset : UCI / Kaggle Heart Disease (target: 0 = no disease, 1 = disease)

Run     : python madaline_heart.py [path/to/heart.csv]

Dataset lookup order
  1. path given on the command line
  2. heart_disease_uci.csv / heart.csv / processed.cleveland.data in the current folder
  3. automatic download of UCI processed.cleveland.data
  4. synthetic fallback data (so the program always runs; results are only a demo)

Requires: numpy, pandas
"""
import os
import sys
import itertools
import urllib.request

import numpy as np
import pandas as pd

RNG_SEED = 42
UCI_URL = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
           "heart-disease/processed.cleveland.data")
UCI_COLS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
            "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"]
NUMERIC = ["age", "trestbps", "chol", "thalach", "oldpeak"]
CATEGORICAL = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]


# ----------------------------------------------------------------------------
# 1. DATA LOADING AND PREPROCESSING
# ----------------------------------------------------------------------------
def _standardise(df):
    """Map Kaggle (heart_disease_data / heart_disease_uci) or UCI layouts to one schema."""
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    rename = {"thalch": "thalach", "num": "target", "condition": "target"}
    df = df.rename(columns=rename)
    if "target" not in df.columns:
        raise ValueError("No target column (target/num/condition) found.")
    df["target"] = (pd.to_numeric(df["target"], errors="coerce") > 0).astype(int)

    # Kaggle heart_disease_uci.csv stores categories as text -> encode to codes
    for c in CATEGORICAL:
        if c in df.columns and df[c].dtype == object:
            df[c] = df[c].astype("category").cat.codes.replace(-1, np.nan)
        if c in df.columns and df[c].dtype == bool:
            df[c] = df[c].astype(int)
    keep = [c for c in NUMERIC + CATEGORICAL if c in df.columns]
    return df[keep + ["target"]]


def load_dataset(path=None):
    candidates = [path] if path else []
    candidates += ["heart_disease_uci.csv", "heart.csv", "heart_disease_data.csv",
                   "processed.cleveland.data"]
    for p in candidates:
        if p and os.path.exists(p):
            print(f"[data] Loading {p}")
            if p.endswith(".data"):
                df = pd.read_csv(p, header=None, names=UCI_COLS, na_values="?")
            else:
                df = pd.read_csv(p, na_values=["?", ""])
            return _standardise(df), "file: " + p
    try:
        print("[data] Trying to download UCI Cleveland data ...")
        with urllib.request.urlopen(UCI_URL, timeout=10) as r:
            open("processed.cleveland.data", "wb").write(r.read())
        df = pd.read_csv("processed.cleveland.data", header=None,
                         names=UCI_COLS, na_values="?")
        return _standardise(df), "UCI download"
    except Exception as e:
        print(f"[data] Download failed ({type(e).__name__}). Using SYNTHETIC demo data.")
        print("[data] Put heart.csv / heart_disease_uci.csv next to this script for real data.")
        return synthetic_data(), "synthetic"


def synthetic_data(n=303):
    rng = np.random.default_rng(RNG_SEED)
    y = rng.integers(0, 2, n)
    d = pd.DataFrame({
        "age": rng.normal(54 + 5 * y, 8), "trestbps": rng.normal(130 + 6 * y, 17),
        "chol": rng.normal(245 + 12 * y, 50), "thalach": rng.normal(158 - 22 * y, 20),
        "oldpeak": np.abs(rng.normal(0.5 + 1.1 * y, 0.9)),
        "sex": (rng.random(n) < 0.5 + 0.2 * y).astype(int),
        "cp": np.where(rng.random(n) < 0.3 + 0.4 * y, 3, rng.integers(0, 3, n)),
        "fbs": (rng.random(n) < 0.15).astype(int),
        "restecg": rng.integers(0, 3, n),
        "exang": (rng.random(n) < 0.15 + 0.4 * y).astype(int),
        "slope": rng.integers(0, 3, n), "ca": rng.integers(0, 4, n) * (y == 1),
        "thal": np.where(y == 1, rng.choice([1, 2], n), rng.choice([0, 1], n)),
        "target": y})
    return d


def preprocess(df):
    """Impute -> one-hot categoricals -> z-score numerics. Returns X, y (bipolar), names."""
    df = df.copy()
    for c in NUMERIC:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce")
            df[c] = df[c].fillna(df[c].median())
    for c in CATEGORICAL:
        if c in df:
            df[c] = df[c].fillna(df[c].mode().iloc[0])
    cats = [c for c in CATEGORICAL if c in df]
    nums = [c for c in NUMERIC if c in df]
    X = pd.get_dummies(df[cats + nums], columns=cats, dtype=float)
    return X, df["target"].values


class Prepared:
    def __init__(self, df, test_size=0.2, seed=RNG_SEED):
        X, y = preprocess(df)
        self.feature_names = list(X.columns)
        X = X.values.astype(float)
        rng = np.random.default_rng(seed)
        # stratified split
        te = []
        for cls in (0, 1):
            idx = np.where(y == cls)[0]
            rng.shuffle(idx)
            te += list(idx[:int(len(idx) * test_size)])
        te = np.array(sorted(te))
        tr = np.setdiff1d(np.arange(len(y)), te)
        # z-score numeric columns using TRAIN statistics only
        num_idx = [i for i, n in enumerate(self.feature_names) if n in NUMERIC]
        self.mu = X[tr][:, num_idx].mean(0)
        self.sd = X[tr][:, num_idx].std(0) + 1e-9
        X[:, num_idx] = (X[:, num_idx] - self.mu) / self.sd
        self.num_idx = num_idx
        X = np.hstack([X, np.ones((len(X), 1))])        # bias input
        self.feature_names.append("bias")
        self.X, self.y01 = X, y
        self.y = np.where(y == 1, 1, -1)                 # bipolar targets
        self.tr, self.te = tr, te


# ----------------------------------------------------------------------------
# 2. MADALINE
# ----------------------------------------------------------------------------
def sgn(v):
    return np.where(v >= 0, 1, -1)


class Madaline:
    """
    Hidden layer : n ADALINE units (linear net, bipolar sign output)
    Output layer : fixed majority vote (sum of hidden outputs, then sign)
    Training     : Stage 1 - LMS (Widrow-Hoff) on each unit's linear output with
                              random-subset targets for diversity
                   Stage 2 - MRII (Minimal Disturbance): when the vote is wrong, flip
                              the unit(s) whose net is closest to 0 and adapt only those.
    """

    def __init__(self, n_units, n_inputs, lr=0.01, seed=RNG_SEED):
        self.n, self.lr = n_units, lr
        self.rng = np.random.default_rng(seed)
        self.W = self.rng.normal(0, 0.1, (n_units, n_inputs))

    # forward
    def nets(self, X):            return X @ self.W.T                  # (m, n)
    def votes(self, X):           return sgn(self.nets(X))
    def vote_sum(self, X):        return self.votes(X).sum(1)
    def predict(self, X):         return sgn(self.vote_sum(X) + 1e-9)  # tie -> +1 (rare, n odd)

    def fit(self, X, y, epochs=60, lms_epochs=15, verbose=False):
        m = len(X)
        history = []
        # Stage 1: LMS with bootstrap samples so units are not identical
        boot = [self.rng.integers(0, m, m) for _ in range(self.n)]
        for _ in range(lms_epochs):
            for j in range(self.n):
                for i in self.rng.permutation(boot[j]):
                    err = y[i] - X[i] @ self.W[j]
                    self.W[j] += self.lr * err * X[i]
        # Stage 2: MRII
        for ep in range(epochs):
            changes = 0
            for i in self.rng.permutation(m):
                changes += self.mrii_step(X[i], y[i])
            acc = (self.predict(X) == y).mean()
            history.append(acc)
            if verbose:
                print(f"  epoch {ep+1:3d}  acc={acc:.3f}  updates={changes}")
            if changes == 0:
                break
        return history

    def mrii_step(self, x, t, apply=True):
        """One minimal-disturbance step. Returns info dict (or 0/1 count if apply)."""
        z = self.W @ x
        y = sgn(z)
        if np.sign(y.sum() + 1e-9) == t:
            return 0 if apply else None
        wrong = [j for j in np.argsort(np.abs(z)) if y[j] != t]   # smallest |net| first
        chosen = None
        for j in wrong:                                           # single-unit flips
            yy = y.copy(); yy[j] = t
            if np.sign(yy.sum() + 1e-9) == t:
                chosen = [j]; break
        if chosen is None:                                        # pairs
            for a, b in itertools.combinations(wrong, 2):
                yy = y.copy(); yy[[a, b]] = t
                if np.sign(yy.sum() + 1e-9) == t:
                    chosen = [a, b]; break
        if chosen is None:
            return 0 if apply else None
        info = {"units": chosen, "nets_before": z.copy(), "candidates": wrong}
        if apply:
            for j in chosen:
                self.W[j] += self.lr * (t - z[j]) * x             # LMS toward flipped target
            return 1
        return info

    def copy(self):
        c = Madaline(self.n, self.W.shape[1], self.lr)
        c.W = self.W.copy()
        return c


# ----------------------------------------------------------------------------
# helpers
# ----------------------------------------------------------------------------
def metrics(y_true, y_pred):
    tp = int(((y_true == 1) & (y_pred == 1)).sum()); tn = int(((y_true == -1) & (y_pred == -1)).sum())
    fp = int(((y_true == -1) & (y_pred == 1)).sum()); fn = int(((y_true == 1) & (y_pred == -1)).sum())
    acc = (tp + tn) / max(1, len(y_true))
    prec = tp / max(1, tp + fp); rec = tp / max(1, tp + fn)
    f1 = 2 * prec * rec / max(1e-12, prec + rec)
    spec = tn / max(1, tn + fp)
    return dict(tp=tp, tn=tn, fp=fp, fn=fn, acc=acc, prec=prec, rec=rec, f1=f1, spec=spec)


def lab(v):  return "Disease(1)" if v > 0 else "No disease(0)"


def ask_int(msg, lo, hi, default=None):
    while True:
        s = input(f"{msg} [{lo}-{hi}]" + (f" (default {default})" if default is not None else "") + ": ").strip()
        if s == "" and default is not None: return default
        try:
            v = int(s)
            if lo <= v <= hi: return v
        except ValueError:
            pass
        print("  invalid input.")


def pick_patient(D, split="test"):
    pool = D.te if split == "test" else D.tr
    print(f"  Patients are indexed by their dataset row number (test set has {len(pool)} rows).")
    print(f"  e.g. {[int(p) for p in pool[:8]]} ...")
    while True:
        s = input("  Enter patient row number (blank = random test patient): ").strip()
        if s == "": return int(np.random.choice(pool))
        try:
            v = int(s)
            if 0 <= v < len(D.y): return v
        except ValueError:
            pass
        print("  invalid row.")


def describe(D, i):
    parts = []
    for k, n in enumerate(D.feature_names):
        if n in NUMERIC:
            j = D.num_idx.index(k)
            parts.append(f"{n}={D.X[i, k] * D.sd[j] + D.mu[j]:.1f}")
    return ", ".join(parts)


# ----------------------------------------------------------------------------
# 3. MENU OPERATIONS
# ----------------------------------------------------------------------------
class App:
    def __init__(self, df, source):
        self.D = Prepared(df)
        self.source = source
        self.units = 5
        self.model = None
        self.train_model(self.units, verbose=False)

    def train_model(self, n, verbose=True):
        D = self.D
        self.model = Madaline(n, D.X.shape[1])
        h = self.model.fit(D.X[D.tr], D.y[D.tr], verbose=verbose)
        self.units = n
        return h

    # -- 1
    def op1(self):
        D = self.D
        n = ask_int("Number of ADALINE units (odd number recommended)", 1, 51, self.units)
        print(f"\nTraining MADALINE with {n} ADALINE units (LMS + MRII) ...")
        self.train_model(n, verbose=True)
        for name, idx in (("TRAIN", D.tr), ("TEST", D.te)):
            m = metrics(D.y[idx], self.model.predict(D.X[idx]))
            print(f"  {name}: accuracy = {m['acc']*100:.2f}%  ({m['tp']+m['tn']}/{len(idx)})")
        i = int(np.random.choice(D.te))
        p = self.model.predict(D.X[i:i+1])[0]
        print(f"\nSample test patient #{i}: predicted {lab(p)}, actual {lab(D.y[i])}")

    # -- 2
    def op2(self):
        D = self.D
        i = pick_patient(D)
        z = self.model.W @ D.X[i]; v = sgn(z)
        print(f"\nPatient #{i}: {describe(D, i)}")
        print(f"Actual class: {lab(D.y[i])}\n")
        print(f"{'Unit':<6}{'Net (w.x)':>12}{'Vote':>8}   Decision")
        for j in range(self.model.n):
            print(f"A{j+1:<5}{z[j]:>12.4f}{v[j]:>8}   {lab(v[j])}")
        pos, neg = int((v > 0).sum()), int((v < 0).sum())
        print(f"\nVotes  Disease: {pos}   No disease: {neg}   Sum: {pos-neg:+d}")
        final = self.model.predict(D.X[i:i+1])[0]
        print(f"MADALINE (majority) output: {lab(final)}  -> "
              f"{'CORRECT' if final == D.y[i] else 'WRONG'}")

    # -- 3
    def op3(self):
        D = self.D
        wrongs = [int(i) for i in D.tr if self.model.predict(D.X[i:i+1])[0] != D.y[i]]
        wrongs.sort(key=lambda i: abs(self.model.vote_sum(D.X[i:i+1])[0]))  # easiest-to-fix first
        if not wrongs:
            print("No misclassified training patients - network already perfect on training data.")
            return
        print(f"{len(wrongs)} misclassified training patients. First few: {wrongs[:8]}")
        s = input("Enter one of them (blank = first): ").strip()
        i = int(s) if s.isdigit() and int(s) in wrongs else wrongs[0]
        x, t = D.X[i], D.y[i]
        mdl = self.model.copy()
        info = mdl.mrii_step(x, t, apply=False)
        z = mdl.W @ x
        print(f"\nPatient #{i}: target {lab(t)}, vote sum {sgn(z).sum():+d} -> WRONG")
        print("Units voting for the wrong class, ranked by |net| (closest to 0 = least disturbance):")
        for j in np.argsort(np.abs(z)):
            if sgn(z[j]) != t:
                print(f"  A{j+1}: net={z[j]:+.4f}  |net|={abs(z[j]):.4f}")
        if info is None:
            print("No single or paired flip can fix this patient (cannot be corrected by MRII).")
            return
        print(f"\nMinimal-disturbance choice: {['A%d' % (u+1) for u in info['units']]}")
        before = mdl.W.copy()
        mdl.mrii_step(x, t, apply=True)
        for u in info["units"]:
            dw = np.linalg.norm(mdl.W[u] - before[u])
            print(f"  A{u+1}: net {z[u]:+.4f} -> {mdl.W[u]@x:+.4f}   ||dW||={dw:.4f}")
        print(f"  Other units untouched: {int(np.all(np.delete(mdl.W,info['units'],0)==np.delete(before,info['units'],0)))==1}")
        print(f"  New prediction: {lab(mdl.predict(x[None])[0])}")
        # contrast: what if we updated the farthest wrong unit instead?
        far = [j for j in np.argsort(-np.abs(z)) if sgn(z[j]) != t][0]
        d_far = abs(self.lr_step(z[far], t)) * np.linalg.norm(x)
        d_min = abs(self.lr_step(z[info['units'][0]], t)) * np.linalg.norm(x)
        print(f"\nChallenge: updating farthest wrong unit A{far+1} would move weights by "
              f"{d_far:.4f} vs {d_min:.4f} for the chosen one.")

    def lr_step(self, z, t): return self.model.lr * (t - z)

    # -- 4
    def op4(self):
        D, M = self.D, self.model
        V = M.votes(D.X[D.te]); n = M.n
        unan = (np.abs(V.sum(1)) == n)
        print(f"Test patients: {len(D.te)}   Unanimous: {int(unan.sum())}   "
              f"Disagreement: {int((~unan).sum())} ({(~unan).mean()*100:.1f}%)")
        print("\nPairwise disagreement rate between hidden units (%):")
        print("      " + "".join(f"A{j+1:<6}" for j in range(n)))
        for a in range(n):
            print(f"A{a+1:<4} " + "".join(f"{(V[:, a] != V[:, b]).mean()*100:<7.1f}" for b in range(n)))
        dis = np.abs(V.sum(1)) if n else 0
        order = np.argsort(dis)[:5]
        print("\nMost contested test patients (smallest |vote sum|):")
        for k in order:
            i = D.te[k]
            print(f"  #{i}: votes {V[k].tolist()} sum={V[k].sum():+d} actual={lab(D.y[i])}")
        acc_u = (V == D.y[D.te][:, None]).mean(0)
        print("\nPer-unit test accuracy: " + ", ".join(f"A{j+1}={a*100:.1f}%" for j, a in enumerate(acc_u)))
        print(f"Majority accuracy: {(M.predict(D.X[D.te]) == D.y[D.te]).mean()*100:.1f}%")

    # -- 5
    def op5(self):
        D, M = self.D, self.model
        i = pick_patient(D)
        x = D.X[i].copy()
        base = M.predict(x[None])[0]
        print(f"\nPatient #{i}: {describe(D, i)}")
        print(f"Baseline decision: {lab(base)} (vote sum {M.vote_sum(x[None])[0]:+d}); actual {lab(D.y[i])}")
        print("\nSingle-feature perturbation search (standardised units):")
        flips = []
        for k, name in enumerate(D.feature_names[:-1]):
            if name in NUMERIC:
                steps = [-3, -2, -1, -0.5, 0.5, 1, 2, 3]
            else:
                steps = [1 - 2 * x[k]]                   # toggle 0/1 dummy
            best = None
            for s in steps:
                xx = x.copy(); xx[k] += s
                if M.predict(xx[None])[0] != base:
                    if best is None or abs(s) < abs(best): best = s
            if best is not None:
                flips.append((abs(best), name, best))
        flips.sort()
        if not flips:
            print("  No single-feature change flips this decision (robust patient).")
        else:
            for _, name, s in flips[:8]:
                if name in NUMERIC:
                    j = D.num_idx.index(D.feature_names.index(name))
                    print(f"  {name:<12} change {s:+.1f} SD (~{s*D.sd[j]:+.1f} units) flips the decision")
                else:
                    print(f"  {name:<12} toggling this indicator flips the decision")
        # which units flip for the smallest change
        if flips:
            _, name, s = flips[0]
            k = D.feature_names.index(name)
            xx = x.copy(); xx[k] += s
            b, a = sgn(M.W @ x), sgn(M.W @ xx)
            print(f"\nSmallest flip via '{name}': units that changed vote -> "
                  f"{['A%d' % (j+1) for j in np.where(b != a)[0]]}")

    # -- 6
    def op6(self):
        D = self.D
        sizes = [1, 3, 5, 7, 9, 15, 25]
        print(f"{'Units':>6}{'Train acc':>11}{'Test acc':>10}{'Test F1':>9}{'Params':>8}")
        best = (0, None)
        for n in sizes:
            m = Madaline(n, D.X.shape[1]); m.fit(D.X[D.tr], D.y[D.tr])
            tr = (m.predict(D.X[D.tr]) == D.y[D.tr]).mean()
            mt = metrics(D.y[D.te], m.predict(D.X[D.te]))
            print(f"{n:>6}{tr*100:>10.2f}%{mt['acc']*100:>9.2f}%{mt['f1']:>9.3f}{n*D.X.shape[1]:>8}")
            if mt["acc"] > best[0] + 1e-9: best = (mt["acc"], n)
        print(f"\nBest test accuracy: {best[0]*100:.2f}% with {best[1]} ADALINE units.")
        print("Use option 1 to retrain the working network with a different size.")

    # -- 7
    def op7(self):
        D, M = self.D, self.model
        thr = 1  # |vote sum| <= thr  => ambiguous
        idx = np.arange(len(D.y))
        Z = M.nets(D.X); vs = sgn(Z).sum(1)
        conf = np.abs(Z).mean(1)
        amb = idx[np.abs(vs) <= thr]
        near = idx[(np.abs(vs) > thr) & (np.abs(Z).min(1) < 0.05)]
        print(f"Ambiguous records (|vote sum| <= {thr}, i.e. narrow majority): {len(amb)} of {len(idx)}")
        print(f"Additional borderline records (a unit's |net| < 0.05): {len(near)}\n")
        print(f"{'Row':>5}{'Votes':>8}{'AvgNet':>9}{'Pred':>15}{'Actual':>15}  Split")
        order = amb[np.argsort(np.abs(Z[amb]).mean(1))][:15]
        for i in order:
            v = sgn(Z[i])
            print(f"{i:>5}{vs[i]:>+8d}{Z[i].mean():>9.3f}{lab(M.predict(D.X[i:i+1])[0]):>15}"
                  f"{lab(D.y[i]):>15}  {int((v>0).sum())}-{int((v<0).sum())}")
        if len(amb):
            err = np.mean(M.predict(D.X[amb]) != D.y[amb]) * 100
            print(f"\nError rate on ambiguous records: {err:.1f}%  |  overall: "
                  f"{np.mean(M.predict(D.X)!=D.y)*100:.1f}%")
            print("Recommendation: send these records for clinician review.")

    # -- 8
    def op8(self):
        D, M = self.D, self.model
        print("=" * 60)
        print("MADALINE CLASSIFICATION PERFORMANCE REPORT")
        print("=" * 60)
        print(f"Data source : {self.source}")
        print(f"Records     : {len(D.y)}  (train {len(D.tr)}, test {len(D.te)})")
        print(f"Features    : {len(D.feature_names)-1} (+bias) after one-hot + z-score")
        print(f"ADALINE units: {M.n}   learning rate: {M.lr}")
        for name, idx in (("TRAIN", D.tr), ("TEST", D.te)):
            m = metrics(D.y[idx], M.predict(D.X[idx]))
            print(f"\n[{name}] accuracy {m['acc']*100:.2f}%  precision {m['prec']:.3f}  "
                  f"recall {m['rec']:.3f}  specificity {m['spec']:.3f}  F1 {m['f1']:.3f}")
            print("            Pred 0   Pred 1")
            print(f"  Actual 0  {m['tn']:>6}   {m['fp']:>6}")
            print(f"  Actual 1  {m['fn']:>6}   {m['tp']:>6}")
        V = M.votes(D.X[D.te])
        print("\nPer-unit TEST accuracy: " +
              ", ".join(f"A{j+1}={a*100:.1f}%" for j, a in enumerate((V == D.y[D.te][:, None]).mean(0))))
        vs = np.abs(V.sum(1))
        print(f"Unanimous decisions on test set: {(vs == M.n).mean()*100:.1f}%")


def main():
    path = sys.argv[1] if len(sys.argv) > 1 else None
    df, source = load_dataset(path)
    print(f"[data] {len(df)} records, {int(df['target'].sum())} with disease.\n")
    app = App(df, source)
    actions = {"1": app.op1, "2": app.op2, "3": app.op3, "4": app.op4,
               "5": app.op5, "6": app.op6, "7": app.op7, "8": app.op8}
    while True:
        print("\n===== MADALINE HEART DISEASE SYSTEM =====")
        print("1. Heart Disease Classification using MADALINE")
        print("2. ADALINE Unit Voting Analyzer")
        print("3. Minimal Disturbance Weight Update Challenge")
        print("4. Hidden ADALINE Unit Disagreement Analyzer")
        print("5. Patient Decision Flip Analyzer")
        print("6. MADALINE Network Size Challenge")
        print("7. Ambiguous Patient Record Detector")
        print("8. MADALINE Classification Performance Report")
        print("9. Exit")
        try:
            c = input("Choice: ").strip()
        except EOFError:
            break
        if c == "9":
            print("Goodbye."); break
        if c in actions:
            print()
            actions[c]()
        else:
            print("Invalid choice.")


if __name__ == "__main__":
    main()

[data] Trying to download UCI Cleveland data ...
[data] 303 records, 139 with disease.


===== MADALINE HEART DISEASE SYSTEM =====
1. Heart Disease Classification using MADALINE
2. ADALINE Unit Voting Analyzer
3. Minimal Disturbance Weight Update Challenge
4. Hidden ADALINE Unit Disagreement Analyzer
5. Patient Decision Flip Analyzer
6. MADALINE Network Size Challenge
7. Ambiguous Patient Record Detector
8. MADALINE Classification Performance Report
9. Exit
Choice: 1

Number of ADALINE units (odd number recommended) [1-51] (default 5): 2

Training MADALINE with 2 ADALINE units (LMS + MRII) ...
  epoch   1  acc=0.820  updates=37
  epoch   2  acc=0.857  updates=38
  epoch   3  acc=0.893  updates=36
  epoch   4  acc=0.840  updates=37
  epoch   5  acc=0.861  updates=40
  epoch   6  acc=0.807  updates=32
  epoch   7  acc=0.857  updates=43
  epoch   8  acc=0.857  updates=37
  epoch   9  acc=0.881  updates=41
  epoch  10  acc=0.828  updates=39
  epoch  11  acc=0.848  updates=37
  epoch  12  a

In [ ]:
"""
BAM Handwritten Digit <-> Code Association System
-------------------------------------------------
Dataset : Optical Recognition of Handwritten Digits (UCI optdigits, 8x8, values 0-16)

Run     : python bam_digits.py [optdigits.tra | optdigits.tes | digits.csv]

Data lookup order
  1. file given on the command line (64 pixel columns + label, comma separated)
  2. optdigits.tra / optdigits.tes in the current folder
  3. scikit-learn's bundled copy of the UCI optdigits test set (1797 samples, works offline)
  4. download from UCI

Requires: numpy   (scikit-learn optional, used only as an offline data source)
"""
import os
import sys
import urllib.request

import numpy as np

SEED = 7
UCI_BASE = "https://archive.ics.uci.edu/ml/machine-learning-databases/optdigits/"
CODE_LEN = 16
THRESHOLD = 8          # pixel (0..16) >= 8  ->  +1 ink, else -1 background


# ----------------------------------------------------------------------------
# data
# ----------------------------------------------------------------------------
def _from_csv(path):
    a = np.loadtxt(path, delimiter=",")
    return a[:, :64], a[:, 64].astype(int)


def load_data(path=None):
    for p in ([path] if path else []) + ["optdigits.tra", "optdigits.tes"]:
        if p and os.path.exists(p):
            print(f"[data] Loading {p}")
            X, y = _from_csv(p)
            return X, y, "file: " + p
    try:
        from sklearn.datasets import load_digits
        d = load_digits()
        print("[data] Using scikit-learn bundled UCI optdigits (1797 samples).")
        return d.data.astype(float), d.target.astype(int), "UCI optdigits via scikit-learn"
    except Exception:
        pass
    try:
        print("[data] Downloading optdigits.tra from UCI ...")
        urllib.request.urlretrieve(UCI_BASE + "optdigits.tra", "optdigits.tra")
        X, y = _from_csv("optdigits.tra")
        return X, y, "UCI download"
    except Exception as e:
        sys.exit(f"[data] Could not load a dataset ({e}). Put optdigits.tra next to this script.")


def to_bipolar(X):
    return np.where(X >= THRESHOLD, 1, -1).astype(int)


def hadamard(n):
    H = np.array([[1]])
    while H.shape[0] < n:
        H = np.block([[H, H], [H, -H]])
    return H


def make_codes():
    """10 unique, mutually orthogonal 16-bit bipolar codes (rows 1..10 of a Hadamard matrix)."""
    return hadamard(CODE_LEN)[1:11].astype(int)


def sgn(v, prev=None):
    out = np.where(v > 0, 1, -1)
    if prev is not None:                       # zero net input -> keep previous state
        out = np.where(v == 0, prev, out)
    return out


def show(p, title="", mask=None):
    """Print an 8x8 bipolar pattern. '#' = +1, '.' = -1, ' ' = unknown (0)."""
    if title: print(title)
    g = p.reshape(8, 8)
    for r in g:
        print("   " + " ".join("#" if v > 0 else ("." if v < 0 else " ") for v in r))


def side_by_side(a, b, ta, tb):
    print(f"   {ta:<17}{tb}")
    for r1, r2 in zip(a.reshape(8, 8), b.reshape(8, 8)):
        f = lambda r: " ".join("#" if v > 0 else ("." if v < 0 else " ") for v in r)
        print(f"   {f(r1)}      {f(r2)}")


def code_str(c): return "".join("+" if v > 0 else "-" for v in c)


# ----------------------------------------------------------------------------
# BAM
# ----------------------------------------------------------------------------
class BAM:
    """
    Correlation (Hebbian) BAM with bias thresholds.
      Learning : W = sum_p (x_p - mu)^T y_p   (mu = mean stored pattern, i.e. threshold/bias)
      Forward  : y = sgn((x - mu) W)
      Backward : x = sgn(mu + y W^T / (L * k))
    Centering removes the large common "background" component shared by all digits,
    which otherwise makes a plain x^T y BAM favour a single class.
    """
    def __init__(self, codes):
        self.codes = codes
        self.W = np.zeros((64, codes.shape[1]), dtype=float)
        self.mu = np.zeros(64)
        self.n_pairs = 0
        self.scale = 1.0

    def store(self, Xb, labels):
        self.mu = Xb.mean(0)
        self.W = np.zeros((64, self.codes.shape[1]))
        for x, l in zip(Xb, labels):
            self.W += np.outer(x - self.mu, self.codes[l])
        self.n_pairs = len(labels)
        self.scale = max(1.0, (self.n_pairs / 10) * self.codes.shape[1])

    def _center(self, x): return np.where(x == 0, 0.0, x - self.mu)   # 0 = unknown pixel

    def forward(self, x, prev=None):   return sgn(self._center(x) @ self.W, prev)
    def backward(self, y, prev=None):  return sgn(self.mu + (y @ self.W.T) / self.scale, prev)

    def decode(self, y):
        """Nearest stored code (min Hamming distance). Returns (digit, distance)."""
        d = (self.codes != y).sum(1)
        return int(d.argmin()), int(d.min())

    def recall(self, x, max_iter=20, x_known=None):
        """Bidirectional iteration x -> y -> x' -> y' ... until stable (resonance).
        x may contain 0 for unknown pixels; x_known (bool mask) clamps known pixels."""
        x = x.copy()
        y = self.forward(x)
        for it in range(1, max_iter + 1):
            x_new = self.backward(y, prev=np.where(x == 0, 1, x))
            if x_known is not None:
                x_new = np.where(x_known, x, x_new)
            y_new = self.forward(x_new, prev=y)
            if np.array_equal(x_new, x) and np.array_equal(y_new, y):
                return x_new, y_new, it
            x, y = x_new, y_new
        return x, y, max_iter

    def classify(self, x):
        return self.decode(self.forward(x))[0]


# ----------------------------------------------------------------------------
# application
# ----------------------------------------------------------------------------
def ask_int(msg, lo, hi, default=None):
    while True:
        s = input(f"{msg} [{lo}-{hi}]" + (f" (default {default})" if default is not None else "") + ": ").strip()
        if s == "" and default is not None: return default
        try:
            v = int(s)
            if lo <= v <= hi: return v
        except ValueError:
            pass
        print("  invalid input.")


class App:
    def __init__(self, X, y, source):
        self.source = source
        self.rng = np.random.default_rng(SEED)
        self.Xb_all, self.y_all = to_bipolar(X), y
        self.codes = make_codes()
        self.bam = BAM(self.codes)
        self.stored = False
        self.per_class = 0
        self.tr_idx = self.te_idx = None
        self.split(per_class=10)

    # choose stored (train) / unseen (test) samples
    def split(self, per_class):
        tr = []
        for d in range(10):
            idx = np.where(self.y_all == d)[0]
            tr += list(self.rng.permutation(idx)[:per_class])
        self.tr_idx = np.array(sorted(tr))
        self.te_idx = np.setdiff1d(np.arange(len(self.y_all)), self.tr_idx)
        self.per_class = per_class

    def need_store(self):
        if not self.stored:
            print("Nothing stored yet - storing with default settings (option 1) first.")
            self.op1(default=True)

    def acc(self, idx, noise=None, mask_fn=None):
        ok = 0
        for i in idx:
            x = self.Xb_all[i].copy()
            if noise: x = self.add_noise(x, noise)
            ok += self.bam.classify(x) == self.y_all[i]
        return ok / len(idx)

    def add_noise(self, x, frac):
        x = x.copy()
        k = int(round(frac * 64))
        pos = self.rng.choice(64, k, replace=False)
        x[pos] *= -1
        return x

    def pick_sample(self, pool="test"):
        d = ask_int("Digit class", 0, 9)
        idx = self.te_idx if pool == "test" else self.tr_idx
        cand = idx[self.y_all[idx] == d]
        s = input(f"  Sample row number (blank = random {pool} sample of digit {d}): ").strip()
        if s.isdigit() and int(s) < len(self.y_all): return int(s)
        return int(self.rng.choice(cand))

    # -- 1
    def op1(self, default=False):
        k = 10 if default else ask_int("Training pairs to store per digit", 1, 100, 10)
        self.split(k)
        self.bam.store(self.Xb_all[self.tr_idx], self.y_all[self.tr_idx])
        self.stored = True
        print(f"\nOutput codes ({CODE_LEN}-bit bipolar, mutually orthogonal):")
        for d in range(10):
            print(f"  digit {d}: {code_str(self.codes[d])}")
        print(f"\nStored {self.bam.n_pairs} pattern pairs ({k} per digit) with W = sum (x-mu)^T y")
        print(f"Weight matrix shape: {self.bam.W.shape}   (|w| max = {np.abs(self.bam.W).max():.1f})")
        print(f"Recall accuracy on STORED pairs : {self.acc(self.tr_idx)*100:.1f}%")
        print(f"Recall accuracy on UNSEEN digits: {self.acc(self.te_idx)*100:.1f}%  ({len(self.te_idx)} samples)")

    # -- 2
    def op2(self):
        self.need_store()
        i = self.pick_sample("test")
        x = self.Xb_all[i]
        y = self.bam.forward(x)
        d, dist = self.bam.decode(y)
        show(x, f"\nInput pattern (row {i}, true digit {self.y_all[i]}):")
        print(f"\nRecalled code : {code_str(y)}")
        print(f"Nearest code  : digit {d}  (Hamming distance {dist})")
        print(f"Result        : {'CORRECT' if d == self.y_all[i] else 'WRONG'}")
        net = self.bam._center(x) @ self.bam.W
        print("Net input per code bit: " + " ".join(f"{v:+.0f}" for v in net))

    # -- 3
    def op3(self):
        self.need_store()
        d = ask_int("Digit code to present", 0, 9)
        y = self.codes[d]
        x = self.bam.backward(y)
        proto = np.where(self.Xb_all[self.y_all == d].mean(0) > 0, 1, -1)
        print(f"\nCode for digit {d}: {code_str(y)}")
        side_by_side(x, proto, "Reconstructed", "Class mean image")
        match = (x == proto).mean() * 100
        print(f"\nPixel agreement with class mean image: {match:.1f}%")
        print(f"Forward check of the reconstruction decodes to digit {self.bam.classify(x)}")

    # -- 4
    def op4(self):
        self.need_store()
        print(f"{'Noise':>7}{'Stored acc':>12}{'Unseen acc':>12}")
        for f in (0, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50):
            print(f"{f*100:>6.0f}%{self.acc(self.tr_idx, f)*100:>11.1f}%{self.acc(self.te_idx, f)*100:>11.1f}%")
        f = float(input("\nNoise fraction for a visual demo (0-1, blank=0.2): ") or 0.2)
        i = self.pick_sample("train")
        clean = self.Xb_all[i]
        noisy = self.add_noise(clean, f)
        x, y, it = self.bam.recall(noisy)
        d, dist = self.bam.decode(y)
        side_by_side(noisy, x, "Noisy input", "BAM output")
        print(f"\nTrue digit {self.y_all[i]}, recalled digit {d} (distance {dist}), iterations {it}")
        print(f"Pixels flipped by noise: {(noisy != clean).sum()}, still wrong after recall: {(x != clean).sum()}")

    # -- 5
    def op5(self):
        self.need_store()
        print("Hide part of the image:\n 1) top half  2) bottom half  3) left half  4) right half  5) random pixels")
        m = ask_int("Mode", 1, 5, 2)
        keep = np.ones((8, 8), bool)
        if m == 1: keep[:4, :] = False
        elif m == 2: keep[4:, :] = False
        elif m == 3: keep[:, :4] = False
        elif m == 4: keep[:, 4:] = False
        else:
            keep = self.rng.random((8, 8)) > 0.5
        keep = keep.ravel()

        def partial(x): return np.where(keep, x, 0)

        i = self.pick_sample("test")
        x_full = self.Xb_all[i]
        xp = partial(x_full)
        x_out, y, it = self.bam.recall(xp, x_known=keep)
        d, dist = self.bam.decode(y)
        show(xp, f"\nPartial input (row {i}, true digit {self.y_all[i]}); blank = unknown:")
        side_by_side(x_out, x_full, "Completed", "Original")
        hidden = ~keep
        print(f"\nRecalled digit {d} (distance {dist}), iterations {it}")
        print(f"Hidden pixels restored correctly: {(x_out[hidden] == x_full[hidden]).mean()*100:.1f}%")
        ok = np.mean([self.bam.decode(self.bam.recall(partial(self.Xb_all[j]), x_known=keep)[1])[0]
                      == self.y_all[j] for j in self.te_idx[:300]])
        print(f"Digit-identification accuracy with this mask over 300 unseen samples: {ok*100:.1f}%")

    # -- 6
    def op6(self):
        self.need_store()
        for name, idx in (("STORED", self.tr_idx), ("UNSEEN", self.te_idx[:500])):
            stable = same = 0; its = []
            for i in idx:
                x0 = self.Xb_all[i]
                y0 = self.bam.forward(x0)
                x1 = self.bam.backward(y0)
                y1 = self.bam.forward(x1)
                stable += np.array_equal(y0, y1)               # code survives a round trip
                same += self.bam.decode(y0)[0] == self.bam.decode(y1)[0]
                its.append(self.bam.recall(x0)[2])
            n = len(idx)
            print(f"[{name}] round-trip code identical: {stable/n*100:.1f}%  "
                  f"same decoded digit: {same/n*100:.1f}%  mean iterations to resonance: {np.mean(its):.2f}")
        print("\nCode -> pattern -> code consistency for each digit:")
        for d in range(10):
            x = self.bam.backward(self.codes[d]); y = self.bam.forward(x)
            print(f"  digit {d}: {'consistent' if np.array_equal(y, self.codes[d]) else 'INCONSISTENT'}"
                  f"  (hamming drift {(y != self.codes[d]).sum()})")

    # -- 7
    def op7(self):
        print("Capacity test: store k pairs per digit, test on the same unseen set\n")
        old = (self.per_class, self.stored)
        base_te = None
        print(f"{'k/digit':>8}{'pairs':>7}{'stored acc':>12}{'unseen acc':>12}{'noisy20% acc':>14}")
        for k in (1, 2, 3, 5, 10, 20, 40, 80):
            self.split(k)
            self.bam.store(self.Xb_all[self.tr_idx], self.y_all[self.tr_idx])
            print(f"{k:>8}{self.bam.n_pairs:>7}{self.acc(self.tr_idx)*100:>11.1f}%"
                  f"{self.acc(self.te_idx[:400])*100:>11.1f}%{self.acc(self.te_idx[:400], 0.2)*100:>13.1f}%")
        print("\nRandom-code capacity probe (distinct patterns stored with random bipolar codes):")
        for p in (4, 8, 16, 32, 64):
            P = self.rng.choice([-1, 1], (p, 64)); Y = self.rng.choice([-1, 1], (p, CODE_LEN))
            W = sum(np.outer(a, b) for a, b in zip(P, Y))
            rec = np.mean([np.array_equal(sgn(a @ W), b) for a, b in zip(P, Y)])
            print(f"  {p:>3} random pairs -> exact recall {rec*100:5.1f}%")
        self.split(old[0])
        self.bam.store(self.Xb_all[self.tr_idx], self.y_all[self.tr_idx]); self.stored = True
        print(f"\n(Network restored to {old[0]} pairs per digit.)")

    # -- 8
    def op8(self):
        self.need_store()
        C = np.zeros((10, 10), int)
        for i in self.te_idx:
            C[self.y_all[i], self.bam.classify(self.Xb_all[i])] += 1
        print("Confusion matrix (rows = true digit, columns = recalled digit)")
        print("      " + "".join(f"{d:>5}" for d in range(10)))
        for d in range(10):
            print(f"  {d:>3} " + "".join(f"{C[d, e]:>5}" for e in range(10)))
        errs = [(C[a, b], a, b) for a in range(10) for b in range(10) if a != b and C[a, b]]
        errs.sort(reverse=True)
        print("\nTop cross-association errors (true -> recalled):")
        for n, a, b in errs[:6]:
            print(f"  {a} -> {b}: {n} cases   (pattern-prototype overlap "
                  f"{self.proto_overlap(a, b):+.2f}, code Hamming distance {(self.codes[a] != self.codes[b]).sum()})")
        if not errs: print("  none")
        # codes separation
        H = [(self.codes[a] != self.codes[b]).sum() for a in range(10) for b in range(a + 1, 10)]
        print(f"\nCode separation: min Hamming distance {min(H)}, max {max(H)} -> codes are well separated.")
        print("Most error-prone digits are those whose input patterns overlap heavily (e.g. 3/5/8, 4/9, 1/7).")

    def proto_overlap(self, a, b):
        pa = self.Xb_all[self.y_all == a].mean(0); pb = self.Xb_all[self.y_all == b].mean(0)
        return float(np.corrcoef(pa, pb)[0, 1])

    # -- 9
    def op9(self):
        self.need_store()
        print("=" * 62); print("BAM ASSOCIATIVE MEMORY REPORT"); print("=" * 62)
        print(f"Dataset        : {self.source}  ({len(self.y_all)} samples)")
        print(f"Input          : 64 bipolar pixels (>= {THRESHOLD} -> +1, else -1)")
        print(f"Output code    : {CODE_LEN}-bit orthogonal bipolar, 10 classes")
        print(f"Stored pairs   : {self.bam.n_pairs} ({self.per_class} per digit)   W shape {self.bam.W.shape}")
        print(f"Stored recall  : {self.acc(self.tr_idx)*100:.1f}%")
        print(f"Unseen recall  : {self.acc(self.te_idx)*100:.1f}% on {len(self.te_idx)} samples")
        for f in (0.1, 0.2, 0.3):
            print(f"  with {int(f*100)}% pixel noise: {self.acc(self.te_idx[:500], f)*100:.1f}%")
        per = [np.mean([self.bam.classify(self.Xb_all[i]) == d for i in self.te_idx[self.y_all[self.te_idx] == d]])
               for d in range(10)]
        print("Per-digit unseen accuracy: " + "  ".join(f"{d}:{p*100:.0f}%" for d, p in enumerate(per)))
        ok = sum(np.array_equal(self.bam.forward(self.bam.backward(self.codes[d])), self.codes[d]) for d in range(10))
        print(f"Code->pattern->code consistent for {ok}/10 digits")


def main():
    X, y, source = load_data(sys.argv[1] if len(sys.argv) > 1 else None)
    print(f"[data] {len(y)} samples loaded.\n")
    app = App(X, y, source)
    app.op1(default=True)
    ops = {"1": app.op1, "2": app.op2, "3": app.op3, "4": app.op4, "5": app.op5,
           "6": app.op6, "7": app.op7, "8": app.op8, "9": app.op9}
    while True:
        print("\n===== BAM DIGIT-CODE ASSOCIATION SYSTEM =====")
        print("1. Store Digit-Code Pattern Associations")
        print("2. Forward Digit Recall Challenge")
        print("3. Reverse Digit Reconstruction Challenge")
        print("4. Noisy Pattern Association Test")
        print("5. Partial Digit Completion Challenge")
        print("6. Bidirectional Recall Consistency Analyzer")
        print("7. Association Capacity Challenge")
        print("8. Cross-Association Error Detector")
        print("9. BAM Associative Memory Report")
        print("10. Exit")
        try:
            c = input("Choice: ").strip()
        except EOFError:
            break
        if c == "10":
            print("Goodbye."); break
        if c in ops:
            print(); ops[c]()
        else:
            print("Invalid choice.")


if __name__ == "__main__":
    main()

[data] Using scikit-learn bundled UCI optdigits (1797 samples).
[data] 1797 samples loaded.


Output codes (16-bit bipolar, mutually orthogonal):
  digit 0: +-+-+-+-+-+-+-+-
  digit 1: ++--++--++--++--
  digit 2: +--++--++--++--+
  digit 3: ++++----++++----
  digit 4: +-+--+-++-+--+-+
  digit 5: ++----++++----++
  digit 6: +--+-++-+--+-++-
  digit 7: ++++++++--------
  digit 8: +-+-+-+--+-+-+-+
  digit 9: ++--++----++--++

Stored 100 pattern pairs (10 per digit) with W = sum (x-mu)^T y
Weight matrix shape: (64, 16)   (|w| max = 64.0)
Recall accuracy on STORED pairs : 83.0%
Recall accuracy on UNSEEN digits: 70.2%  (1697 samples)

===== BAM DIGIT-CODE ASSOCIATION SYSTEM =====
1. Store Digit-Code Pattern Associations
2. Forward Digit Recall Challenge
3. Reverse Digit Reconstruction Challenge
4. Noisy Pattern Association Test
5. Partial Digit Completion Challenge
6. Bidirectional Recall Consistency Analyzer
7. Association Capacity Challenge
8. Cross-Association Error Detector
9. BAM Associ